# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [4]:
import pandas as pd
import numpy as np
import faiss
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
print(f"Jumlah baris, kolom: {df.shape}")
df.head()

Total entri knowledge base: 59
Jumlah baris, kolom: (59, 4)


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [5]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [7]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [8]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [12]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353508)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.4088098)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [13]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [14]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Jawaban LLM:
Halo! Tentu, saya bantu jawab ya.

Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. Apakah ada hal lain yang bisa saya bantu?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [17]:
test_queries = [
    'Apakah ada diskon untuk member baru?',
    'Bagaimana caranya mendaptkan invoice?',
    'Bagaimana cara reset kode sandi?',
    'Kenapa penjual mengatakan stok habis padahal stok yang tertera di aplikasi masih ada?',
    'Apa yang membuat aplikasi ini lebih unggul daripada aplikasi lain?'
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Apakah ada diskon untuk member baru?
Jawaban: Halo! Ya, pengguna baru biasanya mendapatkan voucher selamat datang yang dapat digunakan pada transaksi pertama. Silakan cek halaman promo setelah pendaftaran untuk informasi lebih lanjut ya.
Sumber teratas: Apakah member baru mendapat diskon khusus?
---
Query: Bagaimana caranya mendaptkan invoice?
Jawaban: Halo! Untuk mendapatkan invoice, Anda bisa mengunduhnya secara otomatis melalui halaman detail pesanan setelah pembayaran Anda berhasil dikonfirmasi. Semoga membantu!
Sumber teratas: Bagaimana cara mendapatkan bukti pembayaran atau invoice?
---
Query: Bagaimana cara reset kode sandi?
Jawaban: Halo! Untuk mengatur ulang kata sandi (password) Anda, silakan klik 'Lupa Password' pada halaman login, masukkan email terdaftar, lalu ikuti instruksi reset password yang dikirim ke email Anda. Semoga membantu!
Sumber teratas: Saya lupa password akun, bagaimana cara mengaturnya ulang?
---
Query: Kenapa penjual mengatakan stok habis padahal st

### Catatan Analisis Relevansi

Berdasarkan hasil pengujian pada lima query di atas, performa retrieval dan respons LLM dapat dianalisis sebagai berikut.

1. **“Apakah ada diskon untuk member baru?” — berhasil di-retrieve dengan baik.** FAQ teratas adalah *“Apakah member baru mendapat diskon khusus?”*. Pertanyaan pengguna menggunakan frasa “diskon untuk member baru”, sedangkan FAQ menggunakan “diskon khusus”; meskipun susunan katanya berbeda, maknanya tetap tertangkap. LLM menjawab bahwa pengguna baru biasanya mendapat voucher selamat datang untuk transaksi pertama. Jawaban ini sejalan dengan hasil FAQ yang ditemukan.

2. **“Bagaimana caranya mendaptkan invoice?” — berhasil di-retrieve dengan baik.** FAQ teratas adalah *“Bagaimana cara mendapatkan bukti pembayaran atau invoice?”*. Kata “invoice” menjadi petunjuk semantik yang kuat walaupun pertanyaan pengguna memiliki variasi penulisan (“mendaptkan”) dan struktur kalimat. LLM menyampaikan bahwa invoice dapat diunduh dari halaman detail pesanan setelah pembayaran dikonfirmasi, sesuai dengan knowledge base.

3. **“Bagaimana cara reset kode sandi?” — berhasil di-retrieve dengan baik.** FAQ teratas adalah *“Saya lupa password akun, bagaimana cara mengaturnya ulang?”*. Sistem berhasil menghubungkan “reset kode sandi” dengan “lupa password” dan “mengaturnya ulang”, yang merupakan parafrasa dengan pilihan kata berbeda. Jawaban LLM mengenai tombol “Lupa Password”, email terdaftar, dan instruksi reset sesuai dengan FAQ.

4. **“Kenapa penjual mengatakan stok habis padahal stok yang tertera di aplikasi masih ada?” — berhasil di-retrieve dengan baik, tetapi LLM menambahkan opsi ekskalasi.** FAQ teratas adalah *“Apakah stok yang tertera di aplikasi selalu update secara real-time?”*. Topik stok yang ditampilkan aplikasi memang berkaitan dengan masalah pengguna, sehingga hasil retrieval relevan. Namun, LLM menambahkan di akhir jawaban bahwa jika informasi ini belum menjawab kendala user secara spesifik maka disarankan menghubungi customer service. Padahal, jawaban LLM sudah cukup untuk menjawab pertanyaan dari user. Akan tetapi, hal ini sebenarnya justru lebih baik karena bisa jadi ke depannya jawaban LLM masih general, tidak menyentuh problem spesifik yang dialami user.

5. **“Apa yang membuat aplikasi ini lebih unggul daripada aplikasi lain?” — hasil retrieval kurang relevan, tetapi LLM menangani ketiadaan informasi dengan baik.** FAQ teratas adalah *“Bagaimana cara membandingkan beberapa produk sekaligus?”*. Keduanya sama-sama mengandung konsep “membandingkan”, tetapi maksudnya berbeda: query menanyakan keunggulan aplikasi dibandingkan aplikasi lain, sedangkan FAQ membahas perbandingan produk. LLM tidak mengarang daftar keunggulan; jawabannya menyatakan bahwa informasi tersebut belum tersedia dan menyarankan pengguna menghubungi customer service. Ini sesuai dengan instruksi pada prompt ketika informasi tidak ditemukan.

**Kesimpulan:** Tiga query pertama menunjukkan retrieval yang baik untuk pertanyaan yang jawabannya tersedia, termasuk pertanyaan dengan parafrasa. Query keempat mendapatkan FAQ yang berkaitan, tetapi LLM menambahkan detail yang tidak didukung FAQ. Query kelima memperlihatkan keterbatasan semantic search dalam membedakan dua maksud yang memiliki kemiripan kata. Pada kasus terakhir, LLM berhasil menyatakan keterbatasan informasi. Untuk pengembangan berikutnya, evaluasi sebaiknya tidak hanya menilai apakah FAQ teratas relevan, tetapi juga memeriksa apakah setiap klaim dalam jawaban benar-benar didukung oleh FAQ yang diambil. Ambang relevansi atau mekanisme abstain dapat dipertimbangkan agar sistem tidak menjawab berdasarkan konteks yang hanya mirip secara dangkal.
